# 🧠 Single Agent Pipeline Project

## Problem Statement
Build a **Single-Agent Smart Assistant** that:
- Understands user queries
- Routes tasks based on intent
- Uses tools when required
- Returns structured JSON output

### The agent should handle:
- Math queries → Calculator Tool
- Keyword extraction → Keyword Tool
- General queries → Direct response

---
### 🛠️ What You Need to Implement
- Agent logic
- Conditional routing
- Tool integration
- Basic error handling

### 🚀 Bonus
- Improve routing
- Add logging
- Add more tools


In [1]:
# 🛠️ TOOL 1: Calculator

def calculator(expression: str) -> str:
    """Evaluate a mathematical expression."""
    try:
        result = str(eval(expression))
        logger.info(f"Calculator: {expression} = {result}")
        return result
    except Exception as e:
        logger.error(f"Calculator error: {e}")
        return "Error in calculation"

In [2]:
# 🛠️ TOOL 2: Keyword Extractor

def extract_keywords(text: str) -> list:
    """Extract keywords from text."""
    try:
        words = text.split()
        keywords = list(set([w.lower() for w in words if len(w) > 4]))
        result = keywords[:5]
        logger.info(f"Keywords extracted: {result}")
        return result
    except Exception as e:
        logger.error(f"Keyword extraction error: {e}")
        return []

In [3]:
# 🛠️ TOOL 3: Sentiment Analyzer

def analyze_sentiment(text: str) -> dict:
    """Simple rule-based sentiment analysis."""
    try:
        positive_words = set(["good", "great", "excellent", "amazing", "love", "happy", "best", "awesome", "fantastic", "wonderful", "nice", "perfect"])
        negative_words = set(["bad", "terrible", "awful", "hate", "worst", "poor", "horrible", "ugly", "sad", "angry", "disappointing", "boring"])
        words = text.lower().split()
        pos_count = sum(1 for w in words if w in positive_words)
        neg_count = sum(1 for w in words if w in negative_words)
        if pos_count > neg_count:
            sentiment = "positive"
        elif neg_count > pos_count:
            sentiment = "negative"
        else:
            sentiment = "neutral"
        result = {"sentiment": sentiment, "positive_score": pos_count, "negative_score": neg_count}
        logger.info(f"Sentiment analysis: {result}")
        return result
    except Exception as e:
        logger.error(f"Sentiment analysis error: {e}")
        return {"sentiment": "unknown", "error": str(e)}

In [4]:
# 🛠️ TOOL 4: Text Summarizer

def summarize_text(text: str) -> str:
    """Simple extractive summarizer - picks top sentences by word frequency."""
    try:
        sentences = text.split('. ')
        if len(sentences) <= 2:
            return text
        word_freq = {}
        for word in text.lower().split():
            if len(word) > 3:
                word_freq[word] = word_freq.get(word, 0) + 1
        scored = []
        for sent in sentences:
            score = sum(word_freq.get(w.lower(), 0) for w in sent.split() if len(w) > 3)
            scored.append((score, sent.strip()))
        scored.sort(reverse=True)
        summary = '. '.join([s for _, s in scored[:2]]) + '.'
        logger.info(f"Summarized text ({len(sentences)} sentences -> 2)")
        return summary
    except Exception as e:
        logger.error(f"Summarizer error: {e}")
        return "Error in summarization"

## 🤖 Implement Agent Logic Below

👉 Use conditional routing:
- If query contains "calculate" → use calculator
- If query contains "keywords" → use keyword extractor
- If query contains "summarize" or "summary" → use summarizer
- If query contains "sentiment" or "sentiment of" → use sentiment analyzer
- Else → general response

In [5]:
# 🤖 AGENT FUNCTION

import os
import logging
from dotenv import load_dotenv
from groq import Groq

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

load_dotenv()
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

def agent(query: str):
    query_lower = query.lower()
    logger.info(f"Received query: {query}")

    try:
        if "calculate" in query_lower:
            expr = query_lower.split("calculate")[-1].strip()
            logger.info(f"Routing to calculator with expr: {expr}")
            return {
                "type": "calculation",
                "result": calculator(expr),
            }
        elif "keyword" in query_lower:
            text = query
            for prefix in ["extract keywords from", "keywords from", "extract keywords"]:
                idx = query_lower.find(prefix)
                if idx != -1:
                    text = query[idx + len(prefix):].strip()
                    break
            logger.info(f"Routing to keyword extractor with text: {text}")
            return {
                "type": "keywords",
                "result": extract_keywords(text),
            }
        elif "summarize" in query_lower or "summary" in query_lower:
            text = query
            for prefix in ["summarize", "summary of", "summarize this:"]:
                idx = query_lower.find(prefix)
                if idx != -1:
                    text = query[idx + len(prefix):].strip()
                    break
            logger.info(f"Routing to summarizer with text: {text[:50]}...")
            return {
                "type": "summary",
                "result": summarize_text(text),
            }
        elif "sentiment" in query_lower:
            text = query
            for prefix in ["sentiment of", "analyze sentiment of", "sentiment"]:
                idx = query_lower.find(prefix)
                if idx != -1:
                    text = query[idx + len(prefix):].strip()
                    break
            logger.info(f"Routing to sentiment analyzer with text: {text}")
            return {
                "type": "sentiment",
                "result": analyze_sentiment(text),
            }
        else:
            logger.info("Routing to general LLM response")
            response = client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[
                    {"role": "system", "content": "You are a helpful smart assistant. Answer concisely."},
                    {"role": "user", "content": query},
                ],
            )
            return {
                "type": "general",
                "result": response.choices[0].message.content,
            }
    except Exception as e:
        logger.error(f"Agent error: {e}")
        return {
            "type": "error",
            "result": f"An error occurred: {str(e)}",
        }

## 📦 Expected Output Format

```
{
  "type": "calculation / keywords / general / error",
  "result": ...
}
```

In [6]:
# 🧪 Test Cases

queries = [
    "Calculate 20 + 5",
    "Extract keywords from Artificial Intelligence is transforming industries",
    "Summarize Machine learning is a subset of artificial intelligence. It allows computers to learn from data. It makes predictions without being explicitly programmed. It is used in many applications.",
    "Sentiment of I love this amazing product, it is great and wonderful",
    "Sentiment of This is terrible and disappointing, I hate it",
    "What is machine learning?"
]

for q in queries:
    print("Query:", q)
    print("Response:", agent(q))
    print("-" * 50)

2026-07-12 22:05:31,233 - INFO - Received query: Calculate 20 + 5


2026-07-12 22:05:31,233 - INFO - Routing to calculator with expr: 20 + 5


2026-07-12 22:05:31,233 - INFO - Calculator: 20 + 5 = 25


2026-07-12 22:05:31,234 - INFO - Received query: Extract keywords from Artificial Intelligence is transforming industries


2026-07-12 22:05:31,234 - INFO - Routing to keyword extractor with text: Artificial Intelligence is transforming industries


...

2026-07-12 22:05:31,236 - INFO - Received query: What is machine learning?


2026-07-12 22:05:31,236 - INFO - Routing to general LLM response


Query: Calculate 20 + 5
Response: {'type': 'calculation', 'result': '25'}
--------------------------------------------------
Query: Extract keywords from Artificial Intelligence is transforming industries
Response: {'type': 'keywords', 'result': ['transforming', 'artificial', 'industries', 'intelligence']}
--------------------------------------------------
Query: Summarize Machine learning is a subset of artificial intelligence. It allows computers to learn from data. It makes predictions without being explicitly programmed. It is used in many applications.
Response: {'type': 'summary', 'result': 'It makes predictions without being explicitly programmed. Machine learning is a subset of artificial intelligence.'}
--------------------------------------------------
Query: Sentiment of I love this amazing product, it is great and wonderful
Response: {'type': 'sentiment', 'result': {'sentiment': 'positive', 'positive_score': 4, 'negative_score': 0}}
-----------------------------------------

2026-07-12 22:05:32,943 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Response: {'type': 'general', 'result': 'Machine learning is a type of artificial intelligence that enables computers to learn from data and improve their performance on a task without being explicitly programmed.'}
--------------------------------------------------


In [7]:
# 🎯 Interactive Mode

import sys
if sys.stdin.isatty():
    while True:
        user_input = input("Enter query (type 'exit' to stop): ")
        if user_input.lower() == "exit":
            break
        print("Response:", agent(user_input))
else:
    print("Skipping interactive mode (non-interactive environment)")

Skipping interactive mode (non-interactive environment)
